#### **ATTENTION!!!**
- **Place each section (1,2...) in one or more cells to keep the notebook organised**
- **Complete the codes in the `TODO` sections**

# Python Library Management System

In [1]:
# Install required libraries (This should already exist in colab)
!pip install requests

import pickle
import sqlite3
import requests

## 1. Data Types and Variables

In [2]:
book_title = "Python Programming"   # string
book_quantity = 5                   # integer
book_price = 29.99                  # float
is_available = True                 # boolean

print(f"Book: {book_title}, Quantity: {book_quantity}, Price: ${book_price}, Available: {is_available}")

Book: Python Programming, Quantity: 5, Price: $29.99, Available: True


## 2. Lists

In [3]:
books = ["Python Basics", "Advanced Python", "Data Science with Python"]

# Add two more books to the list
books.append("Machine Learning with Python")   # append() adds at the end
books.insert(0, "Python for AI")               # insert() adds at a given position (here at the start)

print("Updated book list:", books)

Updated book list: ['Python for AI', 'Python Basics', 'Advanced Python', 'Data Science with Python', 'Machine Learning with Python']


## 3. Loops and range()

In [4]:
print("Available books:")
# Use a for loop and range() to print each book with its index
for i in range(len(books)):
    print(f"{i}: {books[i]}")

Available books:
0: Python for AI
1: Python Basics
2: Advanced Python
3: Data Science with Python
4: Machine Learning with Python


## 4. User Input

In [5]:
new_book = input("Enter a new book title: ").strip()

# Edge cases: empty title or book already in the list
if new_book == "":
    print("No title entered, nothing was added.")
elif new_book in books:
    print(f"'{new_book}' is already in the list.")
else:
    books.append(new_book)
    print(f"'{new_book}' was added.")

print("Updated book list:", books)

Enter a new book title: Deep Learning Basics
'Deep Learning Basics' was added.
Updated book list: ['Python for AI', 'Python Basics', 'Advanced Python', 'Data Science with Python', 'Machine Learning with Python', 'Deep Learning Basics']


## 5. Functions and Conditional Statements

In [6]:
def check_availability(book_name):
    # Return True if the book is in the list, False otherwise
    if book_name in books:
        return True
    else:
        return False

# Test the function
print(f"Is 'Python Basics' available? {check_availability('Python Basics')}")
print(f"Is 'Java Basics' available? {check_availability('Java Basics')}")

Is 'Python Basics' available? True
Is 'Java Basics' available? False


## 6. File I/O

In [7]:
def save_books_to_file(filename):
    with open(filename, 'w') as f:
        for book in books:
            f.write(f"{book}\n")

def read_books_from_file(filename):
    # Read the file line by line and return a list of book titles
    try:
        with open(filename, 'r') as f:
            return [line.strip() for line in f if line.strip() != ""]
    except FileNotFoundError:
        print(f"Error: the file '{filename}' does not exist.")
        return []

# Test file I/O
save_books_to_file("books.txt")
!cat books.txt  # Display file contents in Colab

print("Books read from file:", read_books_from_file("books.txt"))
print("Reading a missing file:", read_books_from_file("missing.txt"))  # test the error case

Python for AI
Python Basics
Advanced Python
Data Science with Python
Machine Learning with Python
Deep Learning Basics


Books read from file: ['Python for AI', 'Python Basics', 'Advanced Python', 'Data Science with Python', 'Machine Learning with Python', 'Deep Learning Basics']
Error: the file 'missing.txt' does not exist.
Reading a missing file: []


## 7. Pickle for serialization

In [8]:
def save_books_pickle(filename):
    with open(filename, 'wb') as f:
        pickle.dump(books, f)

def load_books_pickle(filename):
    # Load the list of books saved with pickle
    try:
        with open(filename, 'rb') as f:
            return pickle.load(f)
    except FileNotFoundError:
        print(f"Error: the file '{filename}' does not exist.")
        return []
    except pickle.UnpicklingError:
        print(f"Error: '{filename}' is not a valid pickle file.")
        return []

# Test pickle functions
save_books_pickle("books.pkl")
print("Books loaded from pickle:", load_books_pickle("books.pkl"))

Books loaded from pickle: ['Python for AI', 'Python Basics', 'Advanced Python', 'Data Science with Python', 'Machine Learning with Python', 'Deep Learning Basics']


## 8. SQLite Database Interaction

In [9]:
def create_books_table():
    conn = sqlite3.connect('library.db')
    cursor = conn.cursor()
    cursor.execute('''CREATE TABLE IF NOT EXISTS books
                      (id INTEGER PRIMARY KEY, title TEXT, quantity INTEGER)''')
    conn.commit()
    conn.close()

def add_book_to_db(title, quantity):
    # Check the values before inserting
    if title.strip() == "" or quantity < 0:
        print("Error: the title cannot be empty and the quantity cannot be negative.")
        return
    try:
        conn = sqlite3.connect('library.db')
        cursor = conn.cursor()
        # "?" placeholders protect against SQL injection
        cursor.execute("INSERT INTO books (title, quantity) VALUES (?, ?)", (title, quantity))
        conn.commit()
        print(f"'{title}' (quantity {quantity}) was added to the database.")
    except sqlite3.Error as e:
        print("Database error:", e)
    finally:
        conn.close()

def show_books_in_db():
    # Print every book stored in the database
    conn = sqlite3.connect('library.db')
    cursor = conn.cursor()
    cursor.execute("SELECT id, title, quantity FROM books")
    rows = cursor.fetchall()
    conn.close()
    for row in rows:
        print(row)

# Test database functions
create_books_table()
add_book_to_db("Python for Beginners", 10)
add_book_to_db("", 3)        # test: empty title
show_books_in_db()

'Python for Beginners' (quantity 10) was added to the database.
Error: the title cannot be empty and the quantity cannot be negative.
(1, 'Python for Beginners', 10)


## 9. Web API Interaction (GitHub API)

In [10]:
def get_python_repos():
    url = "https://api.github.com/search/repositories"
    params = {"q": "language:python", "sort": "stars", "order": "desc"}
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()          # error if the status is not 200
        data = response.json()
        return [repo['name'] for repo in data['items'][:5]]
    except requests.exceptions.RequestException as e:
        print("Error while calling the GitHub API:", e)
        return []

# Test API function
print("Top 5 Python repositories on GitHub:", get_python_repos())

Top 5 Python repositories on GitHub: ['public-apis', 'free-programming-books', 'system-design-primer', 'awesome-python', 'project-based-learning']


## 10. Extra features: remove and update books

In [11]:
def remove_book(book_name):
    # Remove a book from the list if it exists
    if book_name in books:
        books.remove(book_name)
        print(f"'{book_name}' was removed.")
    else:
        print(f"'{book_name}' is not in the list.")

def update_book(old_name, new_name):
    # Replace a book title by a new one
    if old_name in books:
        index = books.index(old_name)
        books[index] = new_name
        print(f"'{old_name}' was renamed to '{new_name}'.")
    else:
        print(f"'{old_name}' is not in the list.")

# Test the extra functions
remove_book("Advanced Python")
remove_book("Java Basics")          # test: book not in the list
update_book("Python Basics", "Python Basics (2nd edition)")
print("Final book list:", books)

print("\nLibrary Management System operations completed.")

'Advanced Python' was removed.
'Java Basics' is not in the list.
'Python Basics' was renamed to 'Python Basics (2nd edition)'.
Final book list: ['Python for AI', 'Python Basics (2nd edition)', 'Data Science with Python', 'Machine Learning with Python', 'Deep Learning Basics']

Library Management System operations completed.
